# Iter-SPIN Healthcare — Step 1: iterative identification on healthcare calibration

Iter-SPIN (Contribution 2) applied to the **healthcare** calibration sets, for the variant comparison on
Contribution 3. Re-scores importance on the progressively suppressed model for `N_MAX` rounds, accumulating
a cumulative coupled set; snapshots at n∈{1,2,3,5,10}. **Identical to `SPIN_iter_step1.ipynb`** (authors'
`get_set_difference_mask`, `|W|·Σ|∇L|`, p=q=5e-7, MLP, grouped scoring, in-place suppression between rounds)
— only the calibration data is the healthcare set. **n=1 == one-shot SPIN**, so `coupled_iter_health_n1.json`
should match the Step-1 `coupled_indices_health.json` (1,163 weights) — a sanity check.

**Why re-run Step 1:** coupled neurons are dataset-specific; the general-domain iter sets do not transfer.

**Inputs (add Step-1/Step-2 output as a Kaggle Dataset input):** `cal_fair_health.csv`, `cal_priv_health.csv`,
`cal_general_health.csv`. Internet ON (model). GPU **T4×2**. ~15–20 min/round (~2.5–3 h for 10 rounds),
**checkpointed per round**. **Outputs:** `coupled_iter_health_n{1,2,3,5,10}.json`, `iter_summary_health.json`.

## Step 0 — config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID='Qwen/Qwen2-7B-Instruct'
NSAMPLES=128; TARGET='mlp'; MAXLEN=1024; LAYER_GROUP=4
P=5e-7; Q=5e-7; SEED=0; DTYPE=torch.float16
N_MAX=10; SNAP=[1,2,3,5,10]
np.random.seed(SEED); torch.manual_seed(SEED)
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
print('config | model', MODEL_ID, '| p=q', P, '| N_MAX', N_MAX, '| snapshots', SNAP)

## Step 1 — healthcare calibration + model

In [ ]:
CF=find('cal_fair_health.csv'); CP=find('cal_priv_health.csv'); CG=find('cal_general_health.csv')
assert CF and CP and CG, 'add cal_{fair,priv,general}_health.csv (Step-1/2 output) as input'
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'})
model.eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV); memline('loaded')

def target_names(m): return [n for n,p in m.named_parameters() if p.dim()==2 and 'weight' in n and TARGET in n]
def make_batches(path):
    d=pd.read_csv(path); out=[]
    for i in range(len(d)):
        ep=tokenizer(str(d['prompt'].iloc[i]),   return_tensors='pt').input_ids
        er=tokenizer(str(d['response'].iloc[i]), return_tensors='pt').input_ids[:,1:]
        inp=torch.cat([ep,er],1)[:,:MAXLEN].to(GEN_DEV); tar=inp.clone(); tar[:,:ep.shape[1]]=-100
        if (tar!=-100).sum()>0: out.append((inp,tar))
    return out
BATCHES={'f':make_batches(CF),'p':make_batches(CP),'g':make_batches(CG)}
print('batches | f',len(BATCHES['f']),'| p',len(BATCHES['p']),'| g',len(BATCHES['g']))

## Step 2 — authors' selection + scoring + suppression helpers (verbatim)

In [ ]:
names=target_names(model); nL=model.config.num_hidden_layers
GROUPS=[list(range(g,min(g+LAYER_GROUP,nL))) for g in range(0,nL,LAYER_GROUP)]
def get_set_difference_mask(p, q, W_metric1, W_metric2):
    top_p=int(p*W_metric1.shape[1]*W_metric1.shape[0]); top_q=int(q*W_metric2.shape[1]*W_metric2.shape[0])
    top_p_indices=torch.topk(W_metric1.flatten(), top_p, largest=True)[1]
    top_q_indices=torch.topk(W_metric2.flatten(), top_q, largest=True)[1]
    unique_p=torch.unique(top_p_indices); unique_q=torch.unique(top_q_indices)
    mask_only_metric1=~torch.isin(unique_p, unique_q); mask_only_metric2=~torch.isin(unique_q, unique_p)
    mask_intersection=~(torch.ones_like(unique_p).bool() & mask_only_metric1 & mask_only_metric2)
    return mask_only_metric1, mask_only_metric2, mask_intersection, unique_q
def coupled_indices_for_matrix(W_general, W_fair, W_priv):
    _, m1, _, u1 = get_set_difference_mask(P, Q, W_general, W_fair)
    _, m2, _, u2 = get_set_difference_mask(P, Q, W_general, W_priv)
    return u1[m1 & m2]
def coupling_set_sizes(W_general, W_fair, W_priv):
    def tks(W, ratio):
        k=int(ratio*W.shape[0]*W.shape[1])
        return set(torch.unique(torch.topk(W.flatten(), k, largest=True)[1]).tolist()) if k>=1 else set()
    Sg=tks(W_general,P); Sf=tks(W_fair,Q); Sp=tks(W_priv,Q); fp=Sf & Sp
    return {'n_fp':len(fp),'n_fp_notg':len(fp-Sg)}
@torch.enable_grad()
def score_group(gnames, batches):
    for n,p in model.named_parameters(): p.requires_grad_(n in gnames)
    acc={n:torch.zeros_like(model.get_parameter(n)) for n in gnames}
    for inp,tar in batches:
        model.zero_grad(set_to_none=True); model(input_ids=inp,labels=tar).loss.backward()
        for n in gnames:
            gr=model.get_parameter(n).grad
            if gr is not None: acc[n]+=gr.detach().abs()
    imp={n:(model.get_parameter(n).detach().abs()*acc[n]).cpu() for n in gnames}
    del acc; gc.collect(); torch.cuda.empty_cache(); return imp
def suppress_cumulative(cum):
    d=dict(model.named_parameters()); nz=0
    for n,idxset in cum.items():
        if n in d and idxset:
            p=d[n]; idx=torch.tensor(sorted(idxset),device=p.device); p.data.view(-1)[idx]=0.0; nz+=len(idxset)
    return nz
print('functions ready | matrices', len(names), '| groups', len(GROUPS))

## Step 3 — iterative loop (checkpointed per round, resumable)

In [ ]:
CK='/kaggle/working/iter_ckpt_health.pt'
if os.path.exists(CK):
    st=torch.load(CK); cum={n:set(v) for n,v in st['cum'].items()}; start_round=st['next_round']; rounds_log=st.get('rounds_log',[])
    reapplied=suppress_cumulative(cum); print('resuming round', start_round, '| cumulative', sum(len(v) for v in cum.values()), '| reapplied', reapplied)
else:
    cum={}; start_round=1; rounds_log=[]
model.gradient_checkpointing_enable(); model.config.use_cache=False
for r in range(start_round, N_MAX+1):
    t0=time.time(); new_this_round=0; rd_fp=0; rd_fpng=0
    for grp in GROUPS:
        gnames=[n for n in names if any(f'.layers.{L}.' in n for L in grp)]
        imp_f=score_group(gnames,BATCHES['f']); imp_p=score_group(gnames,BATCHES['p']); imp_g=score_group(gnames,BATCHES['g'])
        for n in gnames:
            fi=coupled_indices_for_matrix(imp_g[n], imp_f[n], imp_p[n])
            ss=coupling_set_sizes(imp_g[n], imp_f[n], imp_p[n]); rd_fp+=ss['n_fp']; rd_fpng+=ss['n_fp_notg']
            if len(fi):
                s=cum.setdefault(n,set()); before=len(s); s.update(int(x) for x in fi.tolist()); new_this_round+=len(s)-before
        del imp_f,imp_p,imp_g; gc.collect(); torch.cuda.empty_cache()
    nz=suppress_cumulative(cum); tot=sum(len(v) for v in cum.values())
    rounds_log.append({'round':r,'seconds':round(time.time()-t0),'new':int(new_this_round),'cumulative':int(tot),
                       'n_fp':int(rd_fp),'n_fp_notg':int(rd_fpng),'zeroed':int(nz)})
    torch.save({'cum':{n:list(v) for n,v in cum.items()},'rounds_log':rounds_log,'next_round':r+1}, CK)
    json.dump({'model':MODEL_ID,'domain':'healthcare','rounds':rounds_log}, open('/kaggle/working/iter_summary_health.json','w'), indent=1)
    print(f'  round {r}/{N_MAX} ({time.time()-t0:.0f}s) | new {new_this_round} | cumulative {tot} | |N_f n N_p| {rd_fp} (clean {rd_fpng})'); memline()
    if r in SNAP:
        json.dump({'model':MODEL_ID,'domain':'healthcare','variant':'iter-SPIN','round':r,'nsamples':NSAMPLES,
                   'target':TARGET,'p':P,'q':Q,'coupled_count':tot,'coupled':{n:sorted(v) for n,v in cum.items()}},
                  open(f'/kaggle/working/coupled_iter_health_n{r}.json','w'))
        print(f'    snapshot -> coupled_iter_health_n{r}.json ({tot} weights)')
model.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad_(False)
print('\nITER-SPIN HEALTHCARE STEP 1 DONE. Sanity: coupled_iter_health_n1.json should match coupled_indices_health.json (1163).')
print('Next: feed snapshots to the iter healthcare Step 2 (HSIC + ppl + accuracy sweep).')